<center>

# Universidad Nacional Autónoma de México
## Facultad de Ciencias

<hr style="height:3px; background-color:#003366; border:none;"/>

## Inteligencia Artificial
### Compendio de Algoritmos — Lab 4.2

**Fundamentos Matemáticos · Descripción Formal · Pseudocódigo · Código · Buenas Prácticas**

<hr style="height:3px; background-color:#003366; border:none;"/>

</center>

## Índice de Algoritmos

### Aprendizaje Supervisado
1. Regresión Lineal
2. Regresión Logística
3. Support Vector Machine (SVM)
4. K-Nearest Neighbors (K-NN)
5. Árbol de Decisión (CART)
6. Random Forest
7. Gradient Boosting
8. XGBoost
9. Red Neuronal Convolucional (CNN)
10. Q-Learning (Aprendizaje por Refuerzo)

### Aprendizaje No Supervisado
11. K-Means
12. DBSCAN
13. Hierarchical Clustering (Aglomerativo)
14. Isolation Forest

---
# 1. Regresión Lineal

## Fundamentos Matemáticos

El modelo predice una salida continua $\hat{y}$ como combinación lineal de las características:

$$\hat{y} = X\mathbf{w} + b$$

La **función de pérdida** es el Error Cuadrático Medio (MSE):

$$\mathcal{L}(\mathbf{w}, b) = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2$$

Los **gradientes** para descenso de gradiente son:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = \frac{2}{n} X^T (\hat{y} - y) \qquad \frac{\partial \mathcal{L}}{\partial b} = \frac{2}{n}\sum(\hat{y} - y)$$

**Actualización de parámetros:**

$$\mathbf{w} \leftarrow \mathbf{w} - \alpha \frac{\partial \mathcal{L}}{\partial \mathbf{w}} \qquad b \leftarrow b - \alpha \frac{\partial \mathcal{L}}{\partial b}$$

donde $\alpha$ es la **tasa de aprendizaje**.

**Complejidad:** Entrenamiento $O(n \cdot d \cdot T)$, Predicción $O(d)$, donde $T$ = iteraciones.

## Descripción Formal

La Regresión Lineal ajusta un hiperplano en el espacio de características que minimiza la suma de los errores cuadráticos entre las predicciones y los valores reales. Se optimiza mediante **Descenso de Gradiente** (Gradient Descent): se calcula el gradiente del error con respecto a cada parámetro y se actualizan en la dirección opuesta al gradiente.

## Pseudocódigo

```
ALGORITMO: Regresión Lineal con Descenso de Gradiente
=====================================================
INICIALIZAR w ← 0, b ← 0

PARA t en [1 ... T_iteraciones]:
    y_pred ← X · w + b
    dw     ← (1/n) · Xᵀ · (y_pred − y)
    db     ← (1/n) · SUMA(y_pred − y)
    w      ← w − α · dw
    b      ← b − α · db
FIN PARA

FUNCIÓN predecir(X):
    RETORNAR X · w + b
```

In [ ]:
# ============================================================
# 1. REGRESIÓN LINEAL — Implementación desde cero
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


class RegresionLineal:
    """
    Regresión Lineal entrenada con Descenso de Gradiente.

    Parámetros
    ----------
    alpha      : float -- Tasa de aprendizaje.
    iteraciones: int   -- Número de épocas de entrenamiento.
    """

    def __init__(self, alpha: float = 0.01, iteraciones: int = 1000):
        self.alpha       = alpha
        self.iteraciones = iteraciones
        self.w           = None
        self.b           = None
        self.historial_perdida = []

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        """Entrena el modelo ajustando w y b mediante descenso de gradiente."""
        n_muestras, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0

        for _ in range(self.iteraciones):
            y_pred = np.dot(X, self.w) + self.b

            # Gradientes del MSE
            dw = (1 / n_muestras) * np.dot(X.T, (y_pred - y))
            db = (1 / n_muestras) * np.sum(y_pred - y)

            # Actualización de parámetros
            self.w -= self.alpha * dw
            self.b -= self.alpha * db

            # Registrar pérdida para análisis de convergencia
            perdida = np.mean((y_pred - y) ** 2)
            self.historial_perdida.append(perdida)

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Genera predicciones: y_hat = X·w + b"""
        return np.dot(X, self.w) + self.b


if __name__ == "__main__":
    # Datos sintéticos
    np.random.seed(42)
    X, y = make_regression(n_samples=200, n_features=1, noise=15, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Entrenamiento
    modelo = RegresionLineal(alpha=0.01, iteraciones=1000)
    modelo.fit(X_train, y_train)

    # Evaluación
    y_pred = modelo.predict(X_test)
    print(f"MSE  : {mean_squared_error(y_test, y_pred):.2f}")
    print(f"R²   : {r2_score(y_test, y_pred):.4f}")

    # Visualización
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    ax1.scatter(X_test, y_test, color="steelblue", alpha=0.6, label="Datos reales")
    ax1.plot(X_test, y_pred, color="tomato", linewidth=2, label="Predicción")
    ax1.set_title("Regresión Lineal — Ajuste", fontweight="bold")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(modelo.historial_perdida, color="purple")
    ax2.set_title("Convergencia del MSE", fontweight="bold")
    ax2.set_xlabel("Iteración"); ax2.set_ylabel("MSE")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

---
# 2. Regresión Logística

## Fundamentos Matemáticos

Transforma la salida lineal en una **probabilidad** usando la función sigmoide:

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad z = X\mathbf{w} + b$$

La **función de pérdida** es la Entropía Cruzada Binaria (Log-Loss):

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)\right]$$

Los gradientes son idénticos en forma a la Regresión Lineal:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = \frac{1}{n} X^T(\hat{y} - y) \qquad \frac{\partial \mathcal{L}}{\partial b} = \frac{1}{n}\sum(\hat{y} - y)$$

**Umbral de decisión:** $\hat{y} \geq 0.5 \Rightarrow$ Clase 1, de lo contrario Clase 0.

**Complejidad:** Entrenamiento $O(n \cdot d \cdot T)$, Predicción $O(d)$.

## Descripción Formal

La Regresión Logística es un clasificador binario que modela la probabilidad posterior $P(y=1 \mid x)$. A diferencia de la Regresión Lineal, usa la función sigmoide para acotar la salida en $[0,1]$. Se optimiza minimizando la Entropía Cruzada mediante Descenso de Gradiente.

## Pseudocódigo

```
ALGORITMO: Regresión Logística
==============================
INICIALIZAR w ← 0, b ← 0

PARA t en [1 ... T_iteraciones]:
    z      ← X · w + b
    y_pred ← σ(z) = 1 / (1 + exp(−z))
    dw     ← (1/n) · Xᵀ · (y_pred − y)
    db     ← (1/n) · SUMA(y_pred − y)
    w      ← w − α · dw
    b      ← b − α · db
FIN PARA

FUNCIÓN predecir(X):
    probabilidades ← σ(X·w + b)
    RETORNAR [1 si p >= 0.5 else 0 para p en probabilidades]
```

In [ ]:
# ============================================================
# 2. REGRESIÓN LOGÍSTICA — Implementación desde cero
# ============================================================
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, classification_report


class RegresionLogistica:
    """
    Clasificador binario basado en la función sigmoide,
    optimizado con Descenso de Gradiente sobre la Entropía Cruzada.

    Parámetros
    ----------
    alpha      : float -- Tasa de aprendizaje.
    iteraciones: int   -- Número de épocas.
    """

    def __init__(self, alpha: float = 0.01, iteraciones: int = 1000):
        self.alpha       = alpha
        self.iteraciones = iteraciones
        self.w           = None
        self.b           = None

    def _sigmoid(self, z: np.ndarray) -> np.ndarray:
        """σ(z) = 1 / (1 + e^{-z}) — acota la salida en (0, 1)."""
        return 1 / (1 + np.exp(-z))

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        """Entrena el clasificador mediante descenso de gradiente."""
        n_muestras, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0

        for _ in range(self.iteraciones):
            modelo_lineal = np.dot(X, self.w) + self.b
            y_pred        = self._sigmoid(modelo_lineal)

            # Gradientes de la Entropía Cruzada
            dw = (1 / n_muestras) * np.dot(X.T, (y_pred - y))
            db = (1 / n_muestras) * np.sum(y_pred - y)

            self.w -= self.alpha * dw
            self.b -= self.alpha * db

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Retorna la probabilidad P(y=1|x) para cada muestra."""
        return self._sigmoid(np.dot(X, self.w) + self.b)

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Clasifica usando umbral de decisión 0.5."""
        return (self.predict_proba(X) >= 0.5).astype(int)


if __name__ == "__main__":
    np.random.seed(42)
    X, y = make_classification(n_samples=300, n_features=2, n_redundant=0,
                                n_informative=2, n_clusters_per_class=1, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    modelo = RegresionLogistica(alpha=0.1, iteraciones=1000)
    modelo.fit(X_train, y_train)

    y_pred = modelo.predict(X_test)
    print(f"Exactitud: {accuracy_score(y_test, y_pred)*100:.2f}%")
    print(classification_report(y_test, y_pred, target_names=["Clase 0","Clase 1"]))

    # Curva sigmoide
    z_vals = np.linspace(-10, 10, 300)
    plt.figure(figsize=(6, 4))
    plt.plot(z_vals, 1/(1+np.exp(-z_vals)), color="steelblue", linewidth=2)
    plt.axhline(0.5, color="tomato", linestyle="--", label="Umbral 0.5")
    plt.title("Función Sigmoide", fontweight="bold")
    plt.xlabel("z"); plt.ylabel("σ(z)"); plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

---
# 3. Support Vector Machine (SVM)

## Fundamentos Matemáticos

SVM busca el **hiperplano de máximo margen** que separa dos clases (etiquetas $\{-1, +1\}$):

$$f(x) = \mathbf{w}^T x - b$$

**Función de pérdida Hinge Loss:**

$$\mathcal{L} = \lambda\|\mathbf{w}\|^2 + \frac{1}{n}\sum_{i=1}^{n}\max\left(0,\ 1 - y_i(\mathbf{w}^T x_i - b)\right)$$

**Gradientes** (regla de subgradiente):

$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = \begin{cases} 2\lambda\mathbf{w} & \text{si } y_i(\mathbf{w}^T x_i - b) \geq 1 \\ 2\lambda\mathbf{w} - y_i x_i & \text{en otro caso} \end{cases}$$

El **margen** entre las clases es $\frac{2}{\|\mathbf{w}\|}$. Maximizar el margen equivale a minimizar $\|\mathbf{w}\|^2$.

**Complejidad:** Entrenamiento $O(n^2)$ a $O(n^3)$ (SMO), Predicción $O(d)$.

## Descripción Formal

SVM encuentra el hiperplano que maximiza la distancia mínima (margen) entre el hiperplano y los puntos más cercanos de cada clase (vectores de soporte). El parámetro $\lambda$ controla el balance entre maximizar el margen y minimizar las violaciones.

## Pseudocódigo

```
ALGORITMO: SVM con Descenso de Subgradiente
===========================================
INICIALIZAR w ← 0, b ← 0

PARA t en [1 ... T_iteraciones]:
    PARA cada (x_i, y_i) en (X, y):
        condición ← y_i · (x_i · w − b) >= 1

        SI condición:                           # Punto correctamente clasificado
            w ← w − lr · (2·λ·w)
        SINO:                                   # Violación del margen
            w ← w − lr · (2·λ·w − y_i·x_i)
            b ← b − lr · y_i
    FIN PARA
FIN PARA

FUNCIÓN predecir(X):
    RETORNAR signo(X · w − b)
```

In [ ]:
# ============================================================
# 3. SVM — Implementación desde cero (Descenso de Subgradiente)
# ============================================================
from sklearn.datasets import make_blobs


class SVM:
    """
    Support Vector Machine lineal entrenado con Descenso de Subgradiente.
    Usa la pérdida Hinge y regularización L2 (parámetro lambda_param).

    Parámetros
    ----------
    lr           : float -- Tasa de aprendizaje.
    lambda_param : float -- Fuerza de regularización L2.
    n_iters      : int   -- Número de iteraciones sobre los datos.
    """

    def __init__(self, lr: float = 0.001, lambda_param: float = 0.01, n_iters: int = 1000):
        self.lr           = lr
        self.lambda_param = lambda_param
        self.n_iters      = n_iters
        self.w            = None
        self.b            = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        """Entrena el SVM. Las etiquetas deben ser {−1, +1}."""
        # Convertir etiquetas {0,1} → {−1,+1} si es necesario
        y_ = np.where(y <= 0, -1, 1)
        n_muestras, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0

        for _ in range(self.n_iters):
            for idx, x_i in enumerate(X):
                # Condición de clasificación correcta con margen >= 1
                if y_[idx] * (np.dot(x_i, self.w) - self.b) >= 1:
                    # Solo penalización por regularización
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    # Penalización por violación del margen + regularización
                    self.w -= self.lr * (2 * self.lambda_param * self.w - np.dot(x_i, y_[idx]))
                    self.b -= self.lr * y_[idx]

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Retorna la clase predicha (+1 ó −1) usando el signo de w·x − b."""
        return np.sign(np.dot(X, self.w) - self.b)


if __name__ == "__main__":
    X, y = make_blobs(n_samples=100, n_features=2, centers=2, cluster_std=1.05, random_state=40)
    y = np.where(y == 0, -1, 1)  # Etiquetas SVM: {−1, +1}

    modelo_svm = SVM(lr=0.001, lambda_param=0.01, n_iters=1000)
    modelo_svm.fit(X, y)

    y_pred = modelo_svm.predict(X)
    exactitud = np.mean(y_pred == y)
    print(f"Exactitud en entrenamiento: {exactitud*100:.2f}%")
    print(f"Pesos (w): {modelo_svm.w}")
    print(f"Sesgo (b): {modelo_svm.b:.4f}")

    # Visualización
    plt.figure(figsize=(7, 5))
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", edgecolors="k", s=60)

    ax = plt.gca()
    xlim = ax.get_xlim()
    x1 = np.linspace(xlim[0], xlim[1], 100)
    x2 = (modelo_svm.b - modelo_svm.w[0] * x1) / modelo_svm.w[1]
    plt.plot(x1, x2, "k-", linewidth=2, label="Frontera de Decisión")
    plt.title("SVM — Hiperplano de Máximo Margen", fontweight="bold")
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

---
# 4. K-Nearest Neighbors (K-NN)

## Fundamentos Matemáticos

K-NN es un algoritmo **no paramétrico** (no aprende parámetros explícitos). Para clasificar una muestra $x$, calcula su distancia a todos los puntos de entrenamiento y vota entre los $k$ más cercanos.

**Distancia Euclidiana ($L_2$):**

$$d(x, x') = \sqrt{\sum_{j=1}^{d}(x_j - x'_j)^2} = \|x - x'\|_2$$

**Distancia Manhattan ($L_1$):**

$$d(x, x') = \sum_{j=1}^{d}|x_j - x'_j|$$

**Regla de clasificación (votación de pluralidad):**

$$\hat{y} = \underset{c}{\arg\max}\sum_{i \in \mathcal{N}_k(x)} \mathbf{1}[y_i = c]$$

donde $\mathcal{N}_k(x)$ son los $k$ vecinos más cercanos a $x$.

**Complejidad:** Entrenamiento $O(1)$, Predicción $O(n \cdot d)$ por muestra (costoso para $n$ grande).

## Descripción Formal

K-NN es un algoritmo **lazy** (perezoso): todo el cómputo ocurre en la predicción, no en el entrenamiento. El "entrenamiento" consiste únicamente en almacenar los datos. El hiperparámetro $k$ controla el balance entre sesgo y varianza: $k$ pequeño → alta varianza; $k$ grande → alto sesgo.

## Pseudocódigo

```
ALGORITMO: K-Nearest Neighbors
================================
FUNCIÓN entrenar(X_train, y_train):
    almacenar X_train, y_train   # O(1) — sin cómputo real

FUNCIÓN predecir(x):
    distancias ← [distancia(x, x_i) para x_i en X_train]
    k_indices  ← argsort(distancias)[:k]
    k_etiquetas← [y_train[i] para i en k_indices]
    RETORNAR clase_más_frecuente(k_etiquetas)
```

In [ ]:
# ============================================================
# 4. K-NEAREST NEIGHBORS — Implementación desde cero
# ============================================================
from collections import Counter


class KNN:
    """
    Clasificador K-Nearest Neighbors no paramétrico.

    Parámetros
    ----------
    k        : int -- Número de vecinos a considerar.
    distancia: str -- 'euclidiana' (L2) o 'manhattan' (L1).
    """

    def __init__(self, k: int = 3, distancia: str = "euclidiana"):
        self.k         = k
        self.distancia = distancia
        self.X_train   = None
        self.y_train   = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        """'Entrenamiento' K-NN: solo almacena los datos. O(1)."""
        self.X_train = X
        self.y_train = y

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predice la clase para cada muestra en X."""
        return np.array([self._predecir_uno(x) for x in X])

    def _predecir_uno(self, x: np.ndarray) -> int:
        """Clasifica una sola muestra por votación de los k vecinos más cercanos."""
        # 1. Calcular distancias a todos los puntos de entrenamiento
        distancias = self._calcular_distancias(x)

        # 2. Obtener índices de los k vecinos más cercanos
        k_indices = np.argsort(distancias)[:self.k]

        # 3. Extraer etiquetas de los k vecinos
        k_etiquetas = [self.y_train[i] for i in k_indices]

        # 4. Votación de pluralidad
        return Counter(k_etiquetas).most_common(1)[0][0]

    def _calcular_distancias(self, x: np.ndarray) -> np.ndarray:
        """Calcula la distancia de x a cada punto de entrenamiento."""
        if self.distancia == "euclidiana":
            return np.sqrt(np.sum((x - self.X_train) ** 2, axis=1))
        elif self.distancia == "manhattan":
            return np.sum(np.abs(x - self.X_train), axis=1)
        else:
            raise ValueError(f"Distancia '{self.distancia}' no soportada.")


if __name__ == "__main__":
    np.random.seed(42)
    X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                n_informative=2, n_clusters_per_class=1, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Comparar distintos valores de k
    valores_k = [1, 3, 5, 7, 11, 15]
    exactitudes = []
    for k in valores_k:
        clf = KNN(k=k)
        clf.fit(X_train, y_train)
        exactitudes.append(accuracy_score(y_test, clf.predict(X_test)))

    mejor_k = valores_k[np.argmax(exactitudes)]
    print(f"Mejor k: {mejor_k}  |  Exactitud: {max(exactitudes)*100:.2f}%")

    plt.figure(figsize=(7, 4))
    plt.plot(valores_k, exactitudes, "o-", color="steelblue")
    plt.title("K-NN — Exactitud vs. número de vecinos k", fontweight="bold")
    plt.xlabel("k"); plt.ylabel("Exactitud"); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

---
# 5. Árbol de Decisión (CART)

## Fundamentos Matemáticos

**Entropía de Shannon** — mide la impureza de un conjunto $S$:

$$H(S) = -\sum_{i=1}^{C} p_i \log_2(p_i)$$

**Información Ganada** — reducción de entropía al dividir con atributo $A$ y umbral $t$:

$$IG(S, A, t) = H(S) - \left(\frac{|S_L|}{|S|}H(S_L) + \frac{|S_R|}{|S|}H(S_R)\right)$$

**Criterio de selección (greedy):**

$$A^*, t^* = \underset{A,\,t}{\arg\max}\ IG(S, A, t)$$

**Complejidad:** Entrenamiento $O(n \cdot d \cdot \log n)$, Predicción $O(\log n)$.

## Descripción Formal

CART construye un árbol binario de forma recursiva y greedy. En cada nodo evalúa todos los cortes posibles para todas las características y elige el que maximiza la Información Ganada. La recursión se detiene cuando la entropía es cero (nodo puro) o se alcanza `max_depth`.

## Pseudocódigo

```
FUNCIÓN construir_árbol(X, y, profundidad):
    SI profundidad >= max_depth O H(y) == 0:
        RETORNAR Hoja(clase_mayoritaria(y))
    mejor_A, mejor_t ← argmax_IG(X, y)
    izq ← construir_árbol(X[x[A]<=t], y[x[A]<=t], prof+1)
    der ← construir_árbol(X[x[A]>t],  y[x[A]>t],  prof+1)
    RETORNAR Nodo(A, t, izq, der)
```

In [ ]:
# ============================================================
# 5. ÁRBOL DE DECISIÓN (CART) — Implementación desde cero
# ============================================================

class Nodo:
    """Nodo del árbol: puede ser pregunta (interno) o respuesta (hoja)."""
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature   = feature
        self.threshold = threshold
        self.left      = left
        self.right     = right
        self.value     = value

    def es_hoja(self) -> bool:
        return self.value is not None


class ArbolDecisionCART:
    """
    Árbol de Decisión CART para clasificación.
    Criterio de impureza: Entropía de Shannon + Información Ganada.
    """

    def __init__(self, max_depth: int = 5):
        self.max_depth = max_depth
        self.raiz      = None

    def fit(self, X, y):
        self.raiz = self._construir(X, y, 0)

    def predict(self, X):
        return np.array([self._recorrer(x, self.raiz) for x in X])

    def _construir(self, X, y, prof):
        if prof >= self.max_depth or len(np.unique(y)) == 1:
            return Nodo(value=Counter(y).most_common(1)[0][0])

        mejor_attr, mejor_umbral = self._mejor_corte(X, y)
        mask_izq = X[:, mejor_attr] <= mejor_umbral
        return Nodo(
            feature=mejor_attr, threshold=mejor_umbral,
            left =self._construir(X[ mask_izq], y[ mask_izq], prof+1),
            right=self._construir(X[~mask_izq], y[~mask_izq], prof+1)
        )

    def _mejor_corte(self, X, y):
        mejor_ig, mejor_attr, mejor_umbral = -1, None, None
        for attr in range(X.shape[1]):
            for umbral in np.unique(X[:, attr]):
                ig = self._ig(X[:, attr], y, umbral)
                if ig > mejor_ig:
                    mejor_ig, mejor_attr, mejor_umbral = ig, attr, umbral
        return mejor_attr, mejor_umbral

    def _ig(self, col, y, umbral):
        mask = col <= umbral
        if mask.sum() == 0 or (~mask).sum() == 0:
            return 0
        n = len(y)
        return (self._entropia(y)
                - (mask.sum()/n) * self._entropia(y[mask])
                - ((~mask).sum()/n) * self._entropia(y[~mask]))

    def _entropia(self, y):
        props = np.bincount(y) / len(y)
        return -np.sum([p * np.log2(p) for p in props if p > 0])

    def _recorrer(self, x, nodo):
        if nodo.es_hoja():
            return nodo.value
        if x[nodo.feature] <= nodo.threshold:
            return self._recorrer(x, nodo.left)
        return self._recorrer(x, nodo.right)


if __name__ == "__main__":
    np.random.seed(42)
    X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                n_informative=2, n_clusters_per_class=1, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    modelo = ArbolDecisionCART(max_depth=4)
    modelo.fit(X_train, y_train)
    print(f"Exactitud: {accuracy_score(y_test, modelo.predict(X_test))*100:.2f}%")

---
# 6. Random Forest

## Fundamentos Matemáticos

Random Forest es un **método de ensamble** que combina $T$ árboles de decisión entrenados con:

1. **Bootstrap Sampling:** cada árbol se entrena con una muestra $\mathcal{D}_t$ de tamaño $n$ con reemplazo:
$$\mathcal{D}_t = \{(x_i, y_i) \mid i \sim \text{Uniforme}(\{1,...,n\},\ \text{con reemplazo})\}$$

2. **Aleatoriedad de características:** en cada nodo se evalúa un subconjunto aleatorio de $m \leq d$ características (típicamente $m = \sqrt{d}$).

3. **Votación mayoritaria** (clasificación):
$$\hat{y} = \underset{c}{\arg\max}\sum_{t=1}^{T} \mathbf{1}[f_t(x) = c]$$

**Varianza del ensamble** (error esperado decrece con $T$):
$$\text{Var}\left(\frac{1}{T}\sum_t f_t\right) = \rho\sigma^2 + \frac{1-\rho}{T}\sigma^2$$

donde $\rho$ es la correlación entre árboles y $\sigma^2$ la varianza individual.

**Complejidad:** Entrenamiento $O(T \cdot n \cdot \sqrt{d} \cdot \log n)$, Predicción $O(T \cdot \log n)$.

## Descripción Formal

Cada árbol ve una muestra diferente de los datos (Bootstrap) y en cada nodo solo considera un subconjunto aleatorio de características. Esto decorrelaciona los árboles y reduce la varianza del ensamble sin aumentar el sesgo.

## Pseudocódigo

```
ALGORITMO: Random Forest
========================
PARA t en [1 ... T_árboles]:
    D_t     ← muestra Bootstrap de D (con reemplazo)
    árbol_t ← construir_árbol(D_t, max_features=sqrt(d))
FIN PARA

FUNCIÓN predecir(x):
    votos ← [árbol_t.predecir(x) para t en 1..T]
    RETORNAR clase_más_frecuente(votos)
```

In [ ]:
# ============================================================
# 6. RANDOM FOREST — Implementación desde cero
# ============================================================

class RandomForest:
    """
    Ensamble de Árboles de Decisión con Bootstrap Sampling.

    Parámetros
    ----------
    n_trees  : int -- Número de árboles en el bosque.
    max_depth: int -- Profundidad máxima de cada árbol.
    """

    def __init__(self, n_trees: int = 10, max_depth: int = 5):
        self.n_trees   = n_trees
        self.max_depth = max_depth
        self.arboles   = []

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        """Entrena n_trees árboles sobre muestras Bootstrap distintas."""
        self.arboles = []
        n_muestras   = X.shape[0]

        for _ in range(self.n_trees):
            # Bootstrap: muestra con reemplazo del mismo tamaño
            indices  = np.random.choice(n_muestras, n_muestras, replace=True)
            X_boot, y_boot = X[indices], y[indices]

            arbol = ArbolDecisionCART(max_depth=self.max_depth)
            arbol.fit(X_boot, y_boot)
            self.arboles.append(arbol)

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Votación mayoritaria entre todos los árboles del bosque."""
        # Predicciones de cada árbol: forma (n_trees, n_muestras)
        predicciones = np.array([arbol.predict(X) for arbol in self.arboles])
        # Transponer → (n_muestras, n_trees) y votar por mayoría
        return np.array([Counter(fila).most_common(1)[0][0] for fila in predicciones.T])


if __name__ == "__main__":
    np.random.seed(42)
    X, y = make_classification(n_samples=300, n_features=4, n_redundant=0,
                                n_informative=4, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Comparar n_trees
    resultados_n = []
    for n in [1, 5, 10, 20, 50]:
        rf = RandomForest(n_trees=n, max_depth=4)
        rf.fit(X_train, y_train)
        resultados_n.append((n, accuracy_score(y_test, rf.predict(X_test))))

    for n, acc in resultados_n:
        print(f"n_trees={n:3d}  →  Exactitud: {acc*100:.2f}%")

---
# 7. Gradient Boosting

## Fundamentos Matemáticos

Gradient Boosting construye un ensamble **secuencial** de modelos débiles (árboles de regresión). Cada árbol $f_m$ aprende a corregir los errores del ensamble anterior.

**Predicción acumulada** tras $M$ árboles:

$$F_M(x) = F_0(x) + \sum_{m=1}^{M} \eta \cdot f_m(x)$$

donde $F_0(x) = \bar{y}$ (media de $y$) y $\eta$ es el *learning rate*.

**Residuos (pseudo-gradientes) para pérdida MSE:**

$$r_i^{(m)} = y_i - F_{m-1}(x_i)$$

Cada árbol $f_m$ se ajusta a los residuos $\{r_i^{(m)}\}$ (regresión sobre el error actual).

**Actualización:**

$$F_m(x) = F_{m-1}(x) + \eta \cdot f_m(x)$$

**Complejidad:** Entrenamiento $O(M \cdot n \cdot d \cdot \log n)$, Predicción $O(M \cdot \log n)$.

## Descripción Formal

En cada iteración, Gradient Boosting calcula los residuos (diferencia entre el target y la predicción actual), entrena un árbol pequeño sobre esos residuos, y suma su predicción —escalada por $\eta$— al modelo acumulado. Con $\eta$ pequeño y muchos árboles, el modelo generaliza mejor.

## Pseudocódigo

```
ALGORITMO: Gradient Boosting (MSE)
===================================
F_0 ← media(y)

PARA m en [1 ... M]:
    residuos ← y − F_{m-1}(X)
    árbol_m  ← entrenar_regresor(X, residuos)
    F_m(X)   ← F_{m-1}(X) + η · árbol_m.predecir(X)
FIN PARA

FUNCIÓN predecir(X):
    RETORNAR F_0 + SUMA(η · árbol_m.predecir(X) para m en 1..M)
```

In [ ]:
# ============================================================
# 7. GRADIENT BOOSTING — Implementación desde cero
# ============================================================

class ArbolRegresor:
    """Árbol de decisión para regresión (valores continuos). Usa MSE como criterio."""

    def __init__(self, max_depth: int = 3):
        self.max_depth = max_depth
        self.raiz      = None

    def fit(self, X, y):
        self.raiz = self._construir(X, y, 0)

    def predict(self, X):
        return np.array([self._recorrer(x, self.raiz) for x in X])

    def _construir(self, X, y, prof):
        if prof >= self.max_depth or len(y) < 2:
            return {"valor": np.mean(y)}  # Hoja: promedio de residuos
        attr, umbral = self._mejor_corte(X, y)
        if attr is None:
            return {"valor": np.mean(y)}
        mask = X[:, attr] <= umbral
        return {"attr": attr, "umbral": umbral,
                "izq": self._construir(X[ mask], y[ mask], prof+1),
                "der": self._construir(X[~mask], y[~mask], prof+1)}

    def _mejor_corte(self, X, y):
        mejor_mse, mejor_attr, mejor_umbral = np.var(y), None, None
        for attr in range(X.shape[1]):
            for umbral in np.unique(X[:, attr]):
                mask = X[:, attr] <= umbral
                if mask.sum() == 0 or (~mask).sum() == 0:
                    continue
                mse = (mask.sum()*np.var(y[mask]) + (~mask).sum()*np.var(y[~mask])) / len(y)
                if mse < mejor_mse:
                    mejor_mse, mejor_attr, mejor_umbral = mse, attr, umbral
        return mejor_attr, mejor_umbral

    def _recorrer(self, x, nodo):
        if "valor" in nodo:
            return nodo["valor"]
        if x[nodo["attr"]] <= nodo["umbral"]:
            return self._recorrer(x, nodo["izq"])
        return self._recorrer(x, nodo["der"])


class GradientBoosting:
    """
    Gradient Boosting para regresión usando árboles de decisión como modelos débiles.

    Parámetros
    ----------
    n_estimators : int   -- Número de árboles.
    learning_rate: float -- Factor de escala para cada árbol (η).
    max_depth    : int   -- Profundidad máxima de cada árbol base.
    """

    def __init__(self, n_estimators: int = 50, learning_rate: float = 0.1, max_depth: int = 3):
        self.n_estimators  = n_estimators
        self.learning_rate = learning_rate
        self.max_depth     = max_depth
        self.arboles       = []
        self.prediccion_inicial = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        """Entrena el ensamble secuencial sobre los residuos acumulados."""
        y = y.astype(float)
        self.prediccion_inicial = np.mean(y)
        F = np.full(len(y), self.prediccion_inicial)

        for _ in range(self.n_estimators):
            residuos = y - F                          # Pseudo-gradientes (MSE)
            arbol    = ArbolRegresor(self.max_depth)
            arbol.fit(X, residuos)
            F += self.learning_rate * arbol.predict(X)
            self.arboles.append(arbol)

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predicción acumulada de todos los árboles."""
        y_pred = np.full(X.shape[0], self.prediccion_inicial)
        for arbol in self.arboles:
            y_pred += self.learning_rate * arbol.predict(X)
        return y_pred


if __name__ == "__main__":
    np.random.seed(42)
    X_reg = np.sort(5 * np.random.rand(100, 1), axis=0)
    y_reg = np.sin(X_reg).ravel() + np.random.normal(0, 0.1, 100)

    gb = GradientBoosting(n_estimators=50, learning_rate=0.2, max_depth=3)
    gb.fit(X_reg, y_reg)

    X_test_reg = np.arange(0, 5, 0.01)[:, None]
    y_hat      = gb.predict(X_test_reg)

    plt.figure(figsize=(8, 4))
    plt.scatter(X_reg, y_reg, alpha=0.5, color="gray", label="Datos reales")
    plt.plot(X_test_reg, y_hat, color="tomato", linewidth=2, label="Gradient Boosting")
    plt.title("Gradient Boosting — Ajuste a función no lineal", fontweight="bold")
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

---
# 8. XGBoost (eXtreme Gradient Boosting)

## Fundamentos Matemáticos

XGBoost extiende Gradient Boosting añadiendo **regularización** para controlar el sobreajuste.

**Función objetivo regularizada:**

$$\mathcal{L}^{(m)} = \sum_{i=1}^{n}\ell(y_i, \hat{y}_i^{(m)}) + \Omega(f_m)$$

$$\Omega(f) = \gamma T + \frac{1}{2}\lambda\sum_{j=1}^{T} w_j^2$$

donde $T$ = número de hojas, $w_j$ = peso de la hoja $j$, $\gamma$ = penalización por hoja, $\lambda$ = regularización L2.

**Valor óptimo regularizado de cada hoja** (derivación de primer y segundo orden):

$$w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}$$

Para pérdida MSE: $g_i = \hat{y}_i - y_i$ (gradiente), $h_i = 1$ (hessiano).

**Complejidad:** Similar a Gradient Boosting pero con optimización más eficiente en la búsqueda de cortes (histogramas).

## Descripción Formal

XGBoost adiciona al Gradient Boosting: (1) regularización L2 sobre los pesos de las hojas, (2) uso de gradientes de segundo orden (hessianos) para aproximación cuadrática de la función objetivo, y (3) técnicas de eficiencia computacional como ordenamiento por histogramas.

## Pseudocódigo

```
ALGORITMO: XGBoost (simplificado)
==================================
F_0 ← media(y)

PARA m en [1 ... M]:
    g_i ← gradiente(y_i, F_{m-1}(x_i))    # primer orden
    h_i ← hessiano(y_i, F_{m-1}(x_i))     # segundo orden
    árbol_m ← construir con hojas w* = -Σg / (Σh + λ)
    F_m ← F_{m-1} + η · árbol_m
FIN PARA
```

In [ ]:
# ============================================================
# 8. XGBOOST — Implementación simplificada desde cero
# ============================================================

class ArbolRegRegl:
    """Árbol regresor con hojas regularizadas por λ (L2)."""

    def __init__(self, max_depth: int = 3, reg_lambda: float = 1.0):
        self.max_depth  = max_depth
        self.reg_lambda = reg_lambda
        self.raiz       = None

    def fit(self, X, y):
        self.raiz = self._construir(X, y, 0)

    def predict(self, X):
        return np.array([self._recorrer(x, self.raiz) for x in X])

    def _construir(self, X, y, prof):
        n = len(y)
        # Hoja regularizada: Σg / (Σh + λ) = Σresiduo / (n + λ)
        if prof >= self.max_depth or n < 2:
            return {"valor": np.sum(y) / (n + self.reg_lambda)}

        attr, umbral = self._mejor_corte(X, y)
        if attr is None:
            return {"valor": np.sum(y) / (n + self.reg_lambda)}

        mask = X[:, attr] <= umbral
        return {"attr": attr, "umbral": umbral,
                "izq": self._construir(X[ mask], y[ mask], prof+1),
                "der": self._construir(X[~mask], y[~mask], prof+1)}

    def _mejor_corte(self, X, y):
        mejor_mse, mejor_attr, mejor_umbral = np.var(y), None, None
        for attr in range(X.shape[1]):
            for umbral in np.unique(X[:, attr]):
                mask = X[:, attr] <= umbral
                if mask.sum() == 0 or (~mask).sum() == 0:
                    continue
                mse = (mask.sum()*np.var(y[mask]) + (~mask).sum()*np.var(y[~mask])) / len(y)
                if mse < mejor_mse:
                    mejor_mse, mejor_attr, mejor_umbral = mse, attr, umbral
        return mejor_attr, mejor_umbral

    def _recorrer(self, x, nodo):
        if "valor" in nodo:
            return nodo["valor"]
        if x[nodo["attr"]] <= nodo["umbral"]:
            return self._recorrer(x, nodo["izq"])
        return self._recorrer(x, nodo["der"])


class XGBoostMini:
    """
    XGBoost simplificado con regularización L2 en las hojas.

    Parámetros
    ----------
    n_estimators : int   -- Número de árboles.
    learning_rate: float -- Factor de escala η.
    max_depth    : int   -- Profundidad máxima por árbol.
    reg_lambda   : float -- Regularización L2 (λ). Mayor λ → hojas más suavizadas.
    """

    def __init__(self, n_estimators=50, learning_rate=0.1, max_depth=3, reg_lambda=1.0):
        self.n_estimators  = n_estimators
        self.learning_rate = learning_rate
        self.max_depth     = max_depth
        self.reg_lambda    = reg_lambda
        self.arboles       = []
        self.F0            = None

    def fit(self, X, y):
        y = y.astype(float)
        self.F0 = np.mean(y)
        F = np.full(len(y), self.F0)

        for _ in range(self.n_estimators):
            residuos = y - F   # gradiente de MSE
            arbol = ArbolRegRegl(self.max_depth, self.reg_lambda)
            arbol.fit(X, residuos)
            F += self.learning_rate * arbol.predict(X)
            self.arboles.append(arbol)

    def predict(self, X):
        y_pred = np.full(X.shape[0], self.F0)
        for arbol in self.arboles:
            y_pred += self.learning_rate * arbol.predict(X)
        return y_pred


if __name__ == "__main__":
    np.random.seed(42)
    X_r = np.sort(5 * np.random.rand(100, 1), axis=0)
    y_r = np.sin(X_r).ravel() + np.random.normal(0, 0.1, 100)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    X_ev = np.arange(0, 5, 0.01)[:, None]

    for ax, lam, titulo in zip(axes, [0.0, 5.0], ["λ=0 (sin regularización)", "λ=5 (regularizado L2)"]):
        modelo = XGBoostMini(n_estimators=50, learning_rate=0.2, max_depth=4, reg_lambda=lam)
        modelo.fit(X_r, y_r)
        ax.scatter(X_r, y_r, alpha=0.4, color="gray")
        ax.plot(X_ev, modelo.predict(X_ev), color="tomato", linewidth=2)
        ax.set_title(f"XGBoost — {titulo}", fontweight="bold")
        ax.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

---
# 9. Red Neuronal Convolucional (CNN)

## Fundamentos Matemáticos

**Operación de convolución** (capa convolucional):

$$(I * K)[i,j] = \sum_{m}\sum_{n} I[i+m,\ j+n] \cdot K[m,n]$$

donde $I$ es la imagen de entrada y $K$ es el filtro/kernel aprendible.

**Activación ReLU:**

$$\text{ReLU}(z) = \max(0, z)$$

**Max Pooling** (reducción espacial de dimensión):

$$P[i,j] = \max_{(m,n) \in \mathcal{R}_{i,j}} I[m,n]$$

**Función Softmax** (capa de salida, $C$ clases):

$$\text{Softmax}(z_k) = \frac{e^{z_k}}{\sum_{j=1}^{C} e^{z_j}}$$

**Pérdida Cross-Entropy:**

$$\mathcal{L} = -\sum_{k=1}^{C} y_k \log(\hat{y}_k)$$

**Complejidad:** Entrenamiento $O\!\left(\sum_l n_{l-1} \cdot s_l^2 \cdot n_l \cdot m_l^2\right)$ donde $s_l$ = tamaño del kernel, $m_l$ = tamaño del mapa de activación.

## Descripción Formal

Una CNN procesa imágenes aplicando filtros convolucionales que detectan patrones espaciales locales (bordes, texturas), seguidos de Max Pooling para reducir la dimensionalidad y capas densas para la clasificación final. Los filtros se aprenden mediante backpropagation.

## Pseudocódigo

```
FORWARD PASS:
  out_conv ← Convolución(imagen, filtros)
  out_relu ← ReLU(out_conv)
  out_pool ← MaxPooling(out_relu)
  y_pred   ← Softmax(Densa(Flatten(out_pool)))

BACKWARD PASS:
  grad_densa ← dL/dSoftmax
  grad_pool  ← MaxPooling.backward(grad_densa)
  grad_conv  ← Convolución.backward(grad_pool) → actualizar filtros
```

In [ ]:
# ============================================================
# 9. CNN — Implementación didáctica desde cero (NumPy)
# ============================================================

class CapaConvolucion:
    """
    Capa convolucional 2D.
    Aprende filtros mediante backpropagation (descenso de gradiente).
    """

    def __init__(self, num_filtros: int, tam_kernel: int):
        self.num_filtros = num_filtros
        self.tam_kernel  = tam_kernel
        # Inicialización normalizada por el tamaño del kernel
        self.filtros = np.random.randn(num_filtros, tam_kernel, tam_kernel) / (tam_kernel**2)

    def _regiones(self, imagen):
        """Generador: extrae regiones de tamaño kernel de la imagen."""
        h, w = imagen.shape
        for i in range(h - self.tam_kernel + 1):
            for j in range(w - self.tam_kernel + 1):
                yield imagen[i:i+self.tam_kernel, j:j+self.tam_kernel], i, j

    def forward(self, entrada: np.ndarray) -> np.ndarray:
        """Aplica la convolución: salida[i,j,f] = región · filtro_f"""
        self._entrada = entrada
        h, w = entrada.shape
        salida = np.zeros((h - self.tam_kernel + 1, w - self.tam_kernel + 1, self.num_filtros))
        for region, i, j in self._regiones(entrada):
            salida[i, j] = np.sum(region * self.filtros, axis=(1, 2))
        return salida

    def backward(self, d_salida: np.ndarray, lr: float) -> None:
        """Actualiza filtros con el gradiente del error."""
        d_filtros = np.zeros_like(self.filtros)
        for region, i, j in self._regiones(self._entrada):
            for f in range(self.num_filtros):
                d_filtros[f] += region * d_salida[i, j, f]
        self.filtros -= lr * d_filtros


class CapaMaxPooling:
    """Capa de Max Pooling — reduce dimensión espacial a la mitad."""

    def __init__(self, tam_pool: int = 2):
        self.tam_pool = tam_pool

    def _regiones(self, imagen):
        h, w, _ = imagen.shape
        for i in range(h // self.tam_pool):
            for j in range(w // self.tam_pool):
                yield imagen[i*self.tam_pool:(i+1)*self.tam_pool,
                             j*self.tam_pool:(j+1)*self.tam_pool], i, j

    def forward(self, entrada: np.ndarray) -> np.ndarray:
        self._entrada = entrada
        h, w, f = entrada.shape
        salida = np.zeros((h // self.tam_pool, w // self.tam_pool, f))
        for region, i, j in self._regiones(entrada):
            salida[i, j] = np.amax(region, axis=(0, 1))
        return salida

    def backward(self, d_salida: np.ndarray) -> np.ndarray:
        """Solo el píxel máximo recibe el gradiente."""
        d_entrada = np.zeros_like(self._entrada)
        for region, i, j in self._regiones(self._entrada):
            h, w, f = region.shape
            amax    = np.amax(region, axis=(0, 1))
            for i2 in range(h):
                for j2 in range(w):
                    for f2 in range(f):
                        if region[i2, j2, f2] == amax[f2]:
                            d_entrada[i*self.tam_pool+i2, j*self.tam_pool+j2, f2] = d_salida[i,j,f2]
        return d_entrada


class CapaSoftmax:
    """Capa densa con activación Softmax para clasificación multiclase."""

    def __init__(self, tam_entrada: int, tam_salida: int):
        self.pesos  = np.random.randn(tam_entrada, tam_salida) / tam_entrada
        self.biases = np.zeros(tam_salida)

    def forward(self, entrada: np.ndarray) -> np.ndarray:
        self._shape_entrada = entrada.shape
        self._entrada_flat  = entrada.flatten()
        totales    = np.dot(self._entrada_flat, self.pesos) + self.biases
        exp        = np.exp(totales - totales.max())   # Estabilidad numérica
        self._salida = exp / exp.sum()
        return self._salida

    def backward(self, d_salida: np.ndarray, lr: float) -> np.ndarray:
        for i, grad in enumerate(d_salida):
            if grad == 0:
                continue
            t_exp  = np.exp(self._salida)
            S      = t_exp.sum()
            d_t    = -t_exp[i] * t_exp / S**2
            d_t[i] = t_exp[i] * (S - t_exp[i]) / S**2
            d_L_t  = grad * d_t
            self.pesos  -= lr * np.outer(self._entrada_flat, d_L_t)
            self.biases -= lr * d_L_t
            return np.dot(self.pesos, d_L_t).reshape(self._shape_entrada)


if __name__ == "__main__":
    np.random.seed(42)
    imagen  = np.random.rand(28, 28)          # Imagen sintética tipo MNIST
    etiqueta = 5                               # Clase correcta

    conv    = CapaConvolucion(num_filtros=8, tam_kernel=3)
    pool    = CapaMaxPooling(tam_pool=2)
    softmax = CapaSoftmax(tam_entrada=13*13*8, tam_salida=10)

    # Forward pass
    out_conv  = conv.forward(imagen)
    out_pool  = pool.forward(out_conv)
    prediccion = softmax.forward(out_pool)

    loss = -np.log(prediccion[etiqueta] + 1e-9)
    print(f"Pérdida Cross-Entropy inicial : {loss:.4f}")
    print(f"Clase predicha (antes de entrenar): {np.argmax(prediccion)}")

    # Backward pass (un paso)
    grad = np.zeros(10)
    grad[etiqueta] = -1 / (prediccion[etiqueta] + 1e-9)
    grad_sm   = softmax.backward(grad, lr=0.005)
    grad_pool = pool.backward(grad_sm)
    conv.backward(grad_pool, lr=0.005)
    print("✓ Un paso de backpropagation completado.")

---
# 10. Q-Learning (Aprendizaje por Refuerzo)

## Fundamentos Matemáticos

Q-Learning aprende una **función de valor de acción** $Q(s, a)$ que estima la recompensa acumulada esperada al tomar la acción $a$ en el estado $s$.

**Ecuación de Bellman** (objetivo de actualización):

$$Q^*(s, a) = \mathbb{E}\left[r + \gamma \max_{a'} Q^*(s', a') \mid s, a\right]$$

**Regla de actualización de Q-Learning (TD(0)):**

$$Q(s, a) \leftarrow Q(s, a) + \alpha\left[r + \gamma \max_{a'} Q(s', a') - Q(s, a)\right]$$

donde:
- $\alpha \in (0,1]$ — tasa de aprendizaje
- $\gamma \in [0,1]$ — factor de descuento (importancia del futuro)
- $r$ — recompensa inmediata
- $s'$ — siguiente estado

**Política Epsilon-Greedy** (exploración vs explotación):

$$a = \begin{cases} \text{acción aleatoria} & \text{con prob. } \varepsilon \\ \underset{a}{\arg\max}\ Q(s, a) & \text{con prob. } 1-\varepsilon \end{cases}$$

## Descripción Formal

Q-Learning es un algoritmo **off-policy** (aprende la política óptima sin seguirla directamente). Mantiene una tabla $Q$ de tamaño $|S| \times |A|$. A través de la interacción con el entorno, actualiza los valores $Q$ usando la Ecuación de Bellman hasta convergencia.

## Pseudocódigo

```
ALGORITMO: Q-Learning
======================
INICIALIZAR Q(s,a) ← 0 para todo s,a

PARA episodio en [1 ... N]:
    s ← estado_inicial
    MIENTRAS no sea estado_terminal:
        a ← ε-greedy(Q, s)
        r, s' ← ejecutar_acción(a)
        Q(s,a) ← Q(s,a) + α[r + γ·max_a' Q(s',a') − Q(s,a)]
        s ← s'
    FIN MIENTRAS
FIN PARA
```

In [ ]:
# ============================================================
# 10. Q-LEARNING — Implementación desde cero
# ============================================================

class AgenteQLearning:
    """
    Agente de Aprendizaje por Refuerzo con Q-Learning tabular.

    Parámetros
    ----------
    n_estados : int   -- Número de estados del entorno.
    n_acciones: int   -- Número de acciones posibles.
    alpha     : float -- Tasa de aprendizaje.
    gamma     : float -- Factor de descuento (importancia del futuro).
    epsilon   : float -- Probabilidad de exploración aleatoria.
    """

    def __init__(
        self,
        n_estados: int, n_acciones: int,
        alpha: float = 0.1, gamma: float = 0.9, epsilon: float = 0.1
    ):
        self.q_table  = np.zeros((n_estados, n_acciones))
        self.alpha    = alpha
        self.gamma    = gamma
        self.epsilon  = epsilon

    def elegir_accion(self, estado: int) -> int:
        """Política ε-greedy: explora con probabilidad ε, explota en otro caso."""
        if np.random.uniform(0, 1) < self.epsilon:
            return np.random.choice(self.q_table.shape[1])   # Exploración
        return np.argmax(self.q_table[estado])                # Explotación

    def aprender(self, s: int, a: int, r: float, s_siguiente: int) -> None:
        """Actualiza Q(s,a) usando la Ecuación de Bellman."""
        td_objetivo = r + self.gamma * np.max(self.q_table[s_siguiente])
        td_error    = td_objetivo - self.q_table[s, a]
        self.q_table[s, a] += self.alpha * td_error


if __name__ == "__main__":
    np.random.seed(42)

    # Entorno: mundo lineal de 5 estados (0=inicio, 4=meta)
    # Acciones: 0=izquierda, 1=derecha
    agente = AgenteQLearning(n_estados=5, n_acciones=2, alpha=0.1, gamma=0.9, epsilon=0.1)

    recompensas_por_episodio = []

    for episodio in range(2000):
        estado = 0
        recompensa_total = 0

        while estado < 4:
            accion = agente.elegir_accion(estado)
            sig_estado = min(4, estado+1) if accion == 1 else max(0, estado-1)
            recompensa = 10.0 if sig_estado == 4 else -1.0
            agente.aprender(estado, accion, recompensa, sig_estado)
            estado = sig_estado
            recompensa_total += recompensa

        recompensas_por_episodio.append(recompensa_total)

    print("Q-Table aprendida:")
    print(agente.q_table.round(2))

    # Visualización de convergencia
    ventana = 100
    promedio = np.convolve(recompensas_por_episodio, np.ones(ventana)/ventana, mode="valid")
    plt.figure(figsize=(8, 4))
    plt.plot(promedio, color="steelblue")
    plt.title("Q-Learning — Recompensa promedio por episodio", fontweight="bold")
    plt.xlabel("Episodio"); plt.ylabel("Recompensa promedio")
    plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
# 11. K-Means (No Supervisado)

## Fundamentos Matemáticos

K-Means busca $K$ centroides $\{\mu_1,...,\mu_K\}$ que minimizan la **inercia** (suma de distancias cuadradas intra-cluster):

$$\mathcal{J} = \sum_{k=1}^{K}\sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$

**Paso de Asignación** (E-step):

$$c_i = \underset{k}{\arg\min}\ \|x_i - \mu_k\|^2$$

**Paso de Actualización** (M-step):

$$\mu_k = \frac{1}{|C_k|}\sum_{x_i \in C_k} x_i$$

**Convergencia:** el algoritmo converge cuando los centroides no cambian entre iteraciones ($\|\mu_k^{(t)} - \mu_k^{(t-1)}\| = 0$ para todo $k$).

**Complejidad:** $O(n \cdot K \cdot d \cdot T)$ por iteración.

## Descripción Formal

K-Means es un algoritmo iterativo de dos pasos. Comienza con $K$ centroides iniciales (normalmente puntos aleatorios del dataset). En cada iteración: (1) asigna cada punto al centroide más cercano, (2) recalcula cada centroide como el promedio de sus puntos asignados. Repite hasta convergencia.

## Pseudocódigo

```
ALGORITMO: K-Means
==================
INICIALIZAR centroides ← K puntos aleatorios de X

REPETIR:
    PARA cada x_i en X:
        c_i ← argmin_k ||x_i − μ_k||²
    PARA cada k en [1..K]:
        μ_k ← promedio de {x_i : c_i = k}
HASTA QUE centroides no cambien (convergencia)
```

In [ ]:
# ============================================================
# 11. K-MEANS — Implementación desde cero
# ============================================================
from sklearn.datasets import make_blobs


class KMeans:
    """
    Algoritmo K-Means de clustering.

    Parámetros
    ----------
    K         : int -- Número de clusters.
    max_iters : int -- Iteraciones máximas.
    """

    def __init__(self, K: int = 3, max_iters: int = 100):
        self.K         = K
        self.max_iters = max_iters
        self.centroides = None
        self.clusters   = None

    def fit(self, X: np.ndarray) -> None:
        """Ejecuta el algoritmo K-Means hasta convergencia."""
        n_muestras = X.shape[0]
        # Inicialización: K puntos aleatorios del dataset
        idx_iniciales   = np.random.choice(n_muestras, self.K, replace=False)
        self.centroides = X[idx_iniciales].copy()

        for _ in range(self.max_iters):
            # E-step: Asignación
            self.clusters = self._asignar_clusters(X)

            # M-step: Actualizar centroides
            centroides_old   = self.centroides.copy()
            self.centroides  = self._actualizar_centroides(X)

            # Criterio de convergencia
            if self._convergio(centroides_old, self.centroides):
                break

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Asigna cada muestra al centroide más cercano."""
        distancias = np.array([np.linalg.norm(X - c, axis=1) for c in self.centroides])
        return np.argmin(distancias, axis=0)

    def _asignar_clusters(self, X: np.ndarray) -> list:
        clusters = [[] for _ in range(self.K)]
        for idx, x in enumerate(X):
            distancias = [np.linalg.norm(x - c) for c in self.centroides]
            clusters[np.argmin(distancias)].append(idx)
        return clusters

    def _actualizar_centroides(self, X: np.ndarray) -> np.ndarray:
        centroides = np.zeros((self.K, X.shape[1]))
        for k, cluster in enumerate(self.clusters):
            if cluster:
                centroides[k] = np.mean(X[cluster], axis=0)
            else:
                centroides[k] = X[np.random.randint(X.shape[0])]  # Reinicializar cluster vacío
        return centroides

    def _convergio(self, old: np.ndarray, new: np.ndarray) -> bool:
        return np.allclose(old, new)


if __name__ == "__main__":
    np.random.seed(42)
    X_blob, _ = make_blobs(n_samples=300, centers=3, cluster_std=0.6, random_state=42)

    km = KMeans(K=3, max_iters=100)
    km.fit(X_blob)
    etiquetas = km.predict(X_blob)

    plt.figure(figsize=(7, 5))
    colores = ["steelblue", "tomato", "seagreen"]
    for k in range(km.K):
        pts = X_blob[etiquetas == k]
        plt.scatter(pts[:, 0], pts[:, 1], color=colores[k], alpha=0.6, label=f"Cluster {k}")
    plt.scatter(km.centroides[:, 0], km.centroides[:, 1],
                c="black", marker="X", s=200, zorder=5, label="Centroides")
    plt.title("K-Means — Resultado del clustering", fontweight="bold")
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

---
# 12. DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

## Fundamentos Matemáticos

DBSCAN define clusters basándose en la **densidad** local de puntos.

**Definiciones clave:**

- **ε-vecindario** de un punto $p$: $N_\varepsilon(p) = \{q \in D \mid d(p,q) \leq \varepsilon\}$
- **Punto núcleo** (*core point*): $p$ tal que $|N_\varepsilon(p)| \geq \text{minPts}$
- **Punto borde** (*border point*): no es núcleo pero está en $N_\varepsilon$ de un núcleo
- **Ruido** (*noise*): no es núcleo ni borde

**Alcanzabilidad por densidad:** $q$ es alcanzable desde $p$ si existe una cadena de puntos núcleo que los conecta dentro de $\varepsilon$.

**Parámetros críticos:** $\varepsilon$ (radio de vecindad) y $\text{minPts}$ (densidad mínima).

**Complejidad:** $O(n \log n)$ con índice espacial, $O(n^2)$ sin él.

## Descripción Formal

A diferencia de K-Means, DBSCAN no requiere especificar $K$ y puede detectar clusters de **formas arbitrarias** y **puntos de ruido** (outliers). Expande clusters desde puntos núcleo hacia sus vecinos alcanzables.

## Pseudocódigo

```
ALGORITMO: DBSCAN
=================
PARA cada punto p no visitado:
    marcar p como visitado
    vecinos ← N_ε(p)

    SI |vecinos| < minPts:
        etiquetar p como RUIDO
    SINO:
        nuevo cluster C ← {p}
        EXPANDIR_CLUSTER(p, vecinos, C)
FIN PARA

FUNCIÓN EXPANDIR_CLUSTER(p, vecinos, C):
    PARA cada q en vecinos:
        SI q no visitado:
            marcar q visitado
            vecinos_q ← N_ε(q)
            SI |vecinos_q| >= minPts:
                vecinos ← vecinos ∪ vecinos_q
        SI q no en ningún cluster:
            añadir q a C
    FIN PARA
```

In [ ]:
# ============================================================
# 12. DBSCAN — Implementación desde cero
# ============================================================
from sklearn.datasets import make_moons


class DBSCAN:
    """
    Clustering basado en densidad. Detecta formas arbitrarias y ruido.

    Parámetros
    ----------
    eps        : float -- Radio de vecindad ε.
    min_samples: int   -- Mínimo de puntos para ser núcleo.
    """

    def __init__(self, eps: float = 0.5, min_samples: int = 5):
        self.eps         = eps
        self.min_samples = min_samples
        self.labels      = None

    def fit(self, X: np.ndarray) -> np.ndarray:
        """Ejecuta DBSCAN. Retorna etiquetas (−1 = ruido)."""
        n = X.shape[0]
        self.labels = np.full(n, -1)      # -1 = ruido por defecto
        visitado    = np.zeros(n, dtype=bool)
        cluster_id  = 0

        for i in range(n):
            if visitado[i]:
                continue
            visitado[i] = True
            vecinos     = self._vecindario(X, i)

            if len(vecinos) < self.min_samples:
                self.labels[i] = -1   # Ruido
            else:
                self._expandir(X, i, vecinos, cluster_id, visitado)
                cluster_id += 1

        return self.labels

    def _vecindario(self, X: np.ndarray, idx: int) -> np.ndarray:
        """Retorna índices de puntos dentro del radio ε del punto idx."""
        distancias = np.linalg.norm(X - X[idx], axis=1)
        return np.where(distancias <= self.eps)[0]

    def _expandir(self, X, idx, vecinos, cluster_id, visitado):
        """Expande el cluster desde un punto núcleo hacia sus vecinos alcanzables."""
        self.labels[idx] = cluster_id
        cola = list(vecinos)

        while cola:
            p = cola.pop(0)
            if not visitado[p]:
                visitado[p] = True
                vecinos_p   = self._vecindario(X, p)
                if len(vecinos_p) >= self.min_samples:
                    cola.extend(vecinos_p.tolist())
            if self.labels[p] == -1:
                self.labels[p] = cluster_id


if __name__ == "__main__":
    np.random.seed(42)
    X_lunas, _ = make_moons(n_samples=300, noise=0.06, random_state=42)

    db     = DBSCAN(eps=0.2, min_samples=5)
    labels = db.fit(X_lunas)

    etiquetas_unicas = np.unique(labels)
    colores = plt.cm.tab10(np.linspace(0, 1, len(etiquetas_unicas)))

    plt.figure(figsize=(7, 5))
    for k, col in zip(etiquetas_unicas, colores):
        mask  = labels == k
        color = "black" if k == -1 else col
        label = "Ruido" if k == -1 else f"Cluster {k}"
        plt.scatter(X_lunas[mask, 0], X_lunas[mask, 1], c=[color], label=label, s=30)

    plt.title("DBSCAN — Formas no convexas y detección de ruido", fontweight="bold")
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

---
# 13. Hierarchical Clustering (Aglomerativo)

## Fundamentos Matemáticos

El clustering jerárquico aglomerativo construye una jerarquía de clusters fusionando en cada paso los dos clusters más cercanos.

**Criterios de enlace** (*linkage*) para la distancia entre clusters $C_i$ y $C_j$:

- **Single linkage** (mínimo): $d(C_i, C_j) = \min_{x \in C_i, y \in C_j} d(x, y)$
- **Complete linkage** (máximo): $d(C_i, C_j) = \max_{x \in C_i, y \in C_j} d(x, y)$
- **Average linkage**: $d(C_i, C_j) = \frac{1}{|C_i||C_j|}\sum_{x \in C_i}\sum_{y \in C_j} d(x, y)$

El resultado es un **dendrograma** que muestra la jerarquía completa de fusiones.

**Complejidad:** $O(n^3)$ para la versión naive, $O(n^2 \log n)$ con optimizaciones.

## Descripción Formal

Inicia con $n$ clusters (uno por punto). En cada paso fusiona los dos clusters con menor distancia según el criterio de enlace elegido. Continúa hasta quedar con un solo cluster. Se corta el dendrograma a la altura deseada para obtener $K$ clusters.

## Pseudocódigo

```
ALGORITMO: Hierarchical Clustering Aglomerativo
================================================
INICIALIZAR clusters ← [{x_0}, {x_1}, ..., {x_{n-1}}]

MIENTRAS |clusters| > K:
    (i, j) ← par con mínima distancia d(C_i, C_j)
    clusters ← clusters − {C_i, C_j} ∪ {C_i ∪ C_j}
FIN MIENTRAS
```

In [ ]:
# ============================================================
# 13. HIERARCHICAL CLUSTERING — Implementación desde cero
# ============================================================

class HierarchicalClustering:
    """
    Clustering Jerárquico Aglomerativo con criterio Single Linkage.

    Parámetros
    ----------
    n_clusters: int -- Número deseado de clusters finales.
    """

    def __init__(self, n_clusters: int = 3):
        self.n_clusters = n_clusters
        self.labels_    = None

    def fit(self, X: np.ndarray) -> np.ndarray:
        """Ejecuta el clustering aglomerativo y retorna etiquetas."""
        # Inicializar: cada punto es su propio cluster (lista de índices)
        clusters = [[i] for i in range(len(X))]

        while len(clusters) > self.n_clusters:
            dist_min = float("inf")
            par      = (0, 1)

            # Buscar el par de clusters más cercano (Single Linkage)
            for i in range(len(clusters)):
                for j in range(i+1, len(clusters)):
                    d = self._distancia_clusters(X, clusters[i], clusters[j])
                    if d < dist_min:
                        dist_min = d
                        par      = (i, j)

            # Fusionar los dos clusters más cercanos
            i, j = par
            nuevo = clusters[i] + clusters[j]
            clusters = [c for k, c in enumerate(clusters) if k not in (i, j)]
            clusters.append(nuevo)

        # Asignar etiquetas
        self.labels_ = np.zeros(len(X), dtype=int)
        for etiqueta, cluster in enumerate(clusters):
            for idx in cluster:
                self.labels_[idx] = etiqueta

        return self.labels_

    def _distancia_clusters(self, X, c1: list, c2: list) -> float:
        """Single Linkage: distancia mínima entre cualquier par de puntos de c1 y c2."""
        return min(np.linalg.norm(X[i] - X[j]) for i in c1 for j in c2)


if __name__ == "__main__":
    np.random.seed(42)
    X_hc, _ = make_blobs(n_samples=80, centers=3, cluster_std=0.5, random_state=42)

    hc     = HierarchicalClustering(n_clusters=3)
    labels = hc.fit(X_hc)

    plt.figure(figsize=(7, 5))
    plt.scatter(X_hc[:, 0], X_hc[:, 1], c=labels, cmap="tab10", s=60, edgecolors="k")
    plt.title("Hierarchical Clustering Aglomerativo (Single Linkage)", fontweight="bold")
    plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

---
# 14. Isolation Forest

## Fundamentos Matemáticos

Isolation Forest detecta **anomalías** basándose en el principio: las anomalías son más fáciles de *aislar* (separar) que los puntos normales.

**Longitud de trayectoria** $h(x)$: número de divisiones aleatorias necesarias para aislar el punto $x$ en un árbol de aislamiento.

**Factor de normalización** (longitud media de búsqueda en BST):

$$c(n) = 2H(n-1) - \frac{2(n-1)}{n} \approx 2(\ln(n-1) + 0.5772)$$

donde $H(n) = \sum_{k=1}^{n}\frac{1}{k}$ (número armónico).

**Puntuación de anomalía:**

$$s(x, n) = 2^{-\frac{\mathbb{E}[h(x)]}{c(n)}}$$

- $s \to 1$: el punto se aísla rápidamente → **anomalía**
- $s \to 0.5$: longitud normal → punto normal
- $s \to 0$: necesita muchas divisiones → punto muy normal

**Complejidad:** Entrenamiento $O(T \cdot n \cdot \log n)$, Predicción $O(T \cdot \log n)$.

## Descripción Formal

Construye $T$ árboles de aislamiento. Cada árbol selecciona aleatoriamente una característica y un valor de corte aleatorio para dividir los datos. Los puntos anómalos (alejados de la distribución principal) se aislan en pocas divisiones, produciendo trayectorias cortas.

## Pseudocódigo

```
ALGORITMO: Isolation Forest
============================
PARA t en [1 ... T]:
    submuestra ← muestra_aleatoria(X, tamaño=ψ)
    iTree_t    ← construir_árbol_aislamiento(submuestra)
FIN PARA

FUNCIÓN anomaly_score(x):
    h_prom ← promedio de h(x, iTree_t) para t en 1..T
    RETORNAR 2^(−h_prom / c(ψ))
```

In [ ]:
# ============================================================
# 14. ISOLATION FOREST — Implementación desde cero
# ============================================================

def _c_factor(n: int) -> float:
    """Factor de normalización: longitud promedio en BST con n nodos."""
    if n <= 1:
        return 0.0
    return 2.0 * (np.log(n - 1) + 0.5772156649) - (2.0 * (n - 1) / n)


class IsolationTree:
    """Árbol de aislamiento: divide aleatoriamente hasta aislar cada punto."""

    def __init__(self, X: np.ndarray, profundidad: int, limite: int):
        self.tam = len(X)

        if profundidad >= limite or self.tam <= 1:
            self.es_hoja = True
            return

        self.es_hoja = False
        # Seleccionar característica y umbral aleatorios
        self.feature  = np.random.randint(0, X.shape[1])
        val_min = X[:, self.feature].min()
        val_max = X[:, self.feature].max()

        if val_min == val_max:
            self.es_hoja = True
            return

        self.umbral = np.random.uniform(val_min, val_max)

        mask       = X[:, self.feature] < self.umbral
        self.izq   = IsolationTree(X[ mask], profundidad+1, limite)
        self.der   = IsolationTree(X[~mask], profundidad+1, limite)


def _longitud_trayectoria(x: np.ndarray, arbol: IsolationTree, altura: int) -> float:
    """Recorre el árbol y devuelve la longitud de trayectoria h(x)."""
    if arbol.es_hoja:
        return altura + (_c_factor(arbol.tam) if arbol.tam > 1 else 0)
    if x[arbol.feature] < arbol.umbral:
        return _longitud_trayectoria(x, arbol.izq, altura+1)
    return _longitud_trayectoria(x, arbol.der, altura+1)


class IsolationForest:
    """
    Detector de anomalías basado en árboles de aislamiento.

    Parámetros
    ----------
    n_trees    : int -- Número de árboles de aislamiento.
    sample_size: int -- Tamaño de submuestra por árbol (ψ).
    """

    def __init__(self, n_trees: int = 100, sample_size: int = 256):
        self.n_trees     = n_trees
        self.sample_size = sample_size
        self.arboles     = []
        self.limite      = int(np.ceil(np.log2(sample_size)))

    def fit(self, X: np.ndarray) -> None:
        """Construye n_trees árboles de aislamiento sobre submuestras aleatorias."""
        self.arboles = []
        for _ in range(self.n_trees):
            idx   = np.random.choice(len(X), self.sample_size, replace=False)
            arbol = IsolationTree(X[idx], 0, self.limite)
            self.arboles.append(arbol)

    def anomaly_score(self, X: np.ndarray) -> np.ndarray:
        """Calcula el score de anomalía s ∈ (0,1). Valores cercanos a 1 = anomalía."""
        scores = []
        for x in X:
            trayectorias = [_longitud_trayectoria(x, t, 0) for t in self.arboles]
            h_prom = np.mean(trayectorias)
            scores.append(2 ** (-h_prom / _c_factor(self.sample_size)))
        return np.array(scores)


if __name__ == "__main__":
    np.random.seed(42)
    X_normal   = 0.3 * np.random.randn(200, 2)
    X_outliers = np.random.uniform(-4, 4, (15, 2))
    X_todo     = np.vstack([X_normal, X_outliers])

    iforest = IsolationForest(n_trees=100, sample_size=64)
    iforest.fit(X_todo)
    scores = iforest.anomaly_score(X_todo)

    plt.figure(figsize=(7, 5))
    sc = plt.scatter(X_todo[:, 0], X_todo[:, 1], c=scores, cmap="RdYlGn_r", s=60, edgecolors="k", linewidths=0.4)
    plt.colorbar(sc, label="Anomaly Score (→1 = anomalía)")
    plt.title("Isolation Forest — Detección de Anomalías", fontweight="bold")
    plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    print(f"Score promedio normales  : {scores[:200].mean():.4f}")
    print(f"Score promedio anomalías : {scores[200:].mean():.4f}")

---
## Resumen Comparativo de Algoritmos

| # | Algoritmo | Tipo | Supervisado | Criterio / Métrica | Hiperparámetros clave |
|---|-----------|------|-------------|---------------------|----------------------|
| 1 | Regresión Lineal | Regresión | ✅ | MSE | α, iteraciones |
| 2 | Regresión Logística | Clasificación | ✅ | Cross-Entropy | α, iteraciones |
| 3 | SVM | Clasificación | ✅ | Hinge Loss | lr, λ, kernel |
| 4 | K-NN | Clasificación | ✅ | Distancia L2/L1 | k |
| 5 | Árbol de Decisión | Clasificación | ✅ | Entropía / IG | max_depth |
| 6 | Random Forest | Clasificación | ✅ | Votación mayoritaria | n_trees, max_depth |
| 7 | Gradient Boosting | Regresión | ✅ | MSE residual | n_estimators, η, max_depth |
| 8 | XGBoost | Regresión | ✅ | MSE + L2 regularizado | n_estimators, η, λ |
| 9 | CNN | Clasificación | ✅ | Cross-Entropy | filtros, kernel, lr |
| 10 | Q-Learning | Control | ✅ (refuerzo) | Ecuación de Bellman | α, γ, ε |
| 11 | K-Means | Clustering | ❌ | Inercia (SSE) | K |
| 12 | DBSCAN | Clustering | ❌ | Densidad local | ε, minPts |
| 13 | Hierarchical | Clustering | ❌ | Distancia de enlace | n_clusters, linkage |
| 14 | Isolation Forest | Detección anomalías | ❌ | Longitud de trayectoria | n_trees, sample_size |

---
<footer style="text-align:center; font-size:12px; color:gray;">
© 2026 UNAM Facultad de Ciencias — Inteligencia Artificial
</footer>